# 🏥 MedPilot — All-In-One Colab
**PhoWhisper (VinAI) + Qwen2.5 · 1 Colab · 1 Ngrok URL**

> ⚠️ Gặp lỗi **Duplicate registration** → **Runtime → Restart runtime** → chạy lại

| Endpoint | Mô tả |
|---|---|
| `POST /v1/audio/transcriptions` | PhoWhisper STT |
| `POST /v1/chat/completions` | Qwen2.5 LLM |
| `POST /v1/extract` | Medical extraction |
| `GET  /health` | Health check |

In [ ]:
# ══ CELL 1 · Dọn dẹp (chạy TRƯỚC TIÊN) ══════════════════
# ⚠️ KHÔNG dùng pkill python3 — sẽ kill Colab kernel!
import subprocess, os

# Chỉ kill ngrok và uvicorn, KHÔNG kill python
subprocess.run('pkill -f \'[n]grok\'   2>/dev/null || true', shell=True)
subprocess.run('pkill -f \'[u]vicorn\' 2>/dev/null || true', shell=True)

# Giải phóng port
for port in [8000, 4040]:
    subprocess.run(f'fuser -k {port}/tcp 2>/dev/null || true', shell=True)

# Reset ngrok state
os.system('rm -rf ~/.ngrok2 /root/.ngrok2 /tmp/ngrok.yml 2>/dev/null || true')

try:
    from pyngrok import ngrok as _ng
    _ng.kill()
except Exception:
    pass

import time; time.sleep(2)
print('✅ Dọn dẹp xong!')

In [ ]:
# ══ CELL 2 · Cài đặt ══════════════════════════════════════
!pip install vllm transformers fastapi pyngrok uvicorn python-multipart librosa -q
print('✅ Cài đặt xong!')

In [ ]:
# ══ CELL 3 · Load PhoWhisper ══════════════════════════════
import torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor

PHO_MODEL = 'vinai/PhoWhisper-medium'
print(f'🎤 Loading {PHO_MODEL} ...')

pho_processor = WhisperProcessor.from_pretrained(PHO_MODEL)
pho_model = WhisperForConditionalGeneration.from_pretrained(
    PHO_MODEL, torch_dtype=torch.float16
).to('cuda').eval()

print(f'✅ PhoWhisper loaded — {torch.cuda.memory_allocated()//1024**2} MB GPU used')

In [ ]:
# ══ CELL 4 · Load vLLM + Qwen2.5 ════════════════════════
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

LLM_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
print(f'🤖 Loading {LLM_MODEL} ...')

llm = LLM(
    model=LLM_MODEL,
    tensor_parallel_size=1,
    max_model_len=4096,
    gpu_memory_utilization=0.55,
    dtype='float16'
)
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL, trust_remote_code=True)
print(f'✅ Qwen loaded — {torch.cuda.memory_allocated()//1024**2} MB GPU used')

In [ ]:
# ══ CELL 5 · FastAPI app ══════════════════════════════════
import os, tempfile, librosa, torch
from fastapi import FastAPI, HTTPException, UploadFile, File
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import List, Optional

app = FastAPI(title='MedPilot All-In-One', version='4.1')
app.add_middleware(CORSMiddleware,
    allow_origins=['*'], allow_methods=['*'], allow_headers=['*'])

class Msg(BaseModel):     role: str;  content: str
class ChatReq(BaseModel):
    model: str = LLM_MODEL;  messages: List[Msg]
    temperature: float = 0.7;  max_tokens: int = 2048;  stream: bool = False
class ExtractReq(BaseModel):
    transcript: str;  medical_record: Optional[str] = ''

EXTRACT_PROMPT = '''Bạn là trợ lý y khoa. Trích xuất thông tin từ hội thoại.
TRANSCRIPT: {transcript}\nHỒ SƠ CŨ: {medical_record}
Chỉ trả về JSON hợp lệ:
{{"reason_for_visit":"","main_symptoms":[],"onset_time":"","lesion_location":"",
"related_factors":[],"medical_history":[],"previous_treatment":[],
"missing_fields":[],"summary":"","draft_notes":""}}'''


@app.get('/')
def root(): return {'status':'running','stt':PHO_MODEL,'llm':LLM_MODEL}

@app.get('/health')
def health(): return {'status':'healthy','stt':PHO_MODEL,'llm':LLM_MODEL}


@app.post('/v1/audio/transcriptions')
async def transcribe(file: UploadFile = File(...)):
    tmp_path = None
    try:
        data = await file.read()
        if not data: raise HTTPException(400, 'File rỗng')
        ext = os.path.splitext(file.filename or 'audio.webm')[1] or '.webm'
        with tempfile.NamedTemporaryFile(delete=False, suffix=ext) as f:
            f.write(data); tmp_path = f.name
        audio, _ = librosa.load(tmp_path, sr=16000, mono=True)
        inp = pho_processor(audio, sampling_rate=16000, return_tensors='pt').input_features.to('cuda').half()
        with torch.no_grad():
            ids = pho_model.generate(inp, language='vi', task='transcribe', num_beams=5)
        text = pho_processor.batch_decode(ids, skip_special_tokens=True)[0].strip()
        duration = round(len(audio)/16000, 2)
        print(f'✅ PhoWhisper ({duration}s): {text[:80]}')
        return {'text': text, 'language': 'vi', 'duration': duration, 'segments': []}
    except HTTPException: raise
    except Exception as e: raise HTTPException(500, f'PhoWhisper error: {e}')
    finally:
        if tmp_path and os.path.exists(tmp_path): os.unlink(tmp_path)


@app.post('/v1/chat/completions')
async def chat(req: ChatReq):
    try:
        msgs = [{'role':m.role,'content':m.content} for m in req.messages]
        tokens = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, tokenize=True)
        params = SamplingParams(temperature=req.temperature,
                                max_tokens=min(req.max_tokens,2048), stop=['<|im_end|>'])
        out = llm.generate([tokens], params)[0].outputs[0].text
        return {'id':'chatcmpl-1','object':'chat.completion','model':req.model,
                'choices':[{'index':0,'message':{'role':'assistant','content':out},'finish_reason':'stop'}],
                'usage':{'prompt_tokens':len(tokens),'completion_tokens':len(tokenizer.encode(out)),
                         'total_tokens':len(tokens)+len(tokenizer.encode(out))}}
    except Exception as e: raise HTTPException(500, str(e))


@app.post('/v1/extract')
async def extract(req: ExtractReq):
    try:
        prompt = EXTRACT_PROMPT.format(transcript=req.transcript, medical_record=req.medical_record or 'Không có')
        tokens = tokenizer.apply_chat_template([{'role':'user','content':prompt}], add_generation_prompt=True, tokenize=True)
        out = llm.generate([tokens], SamplingParams(temperature=0.2, max_tokens=1500))[0].outputs[0].text
        return {'extraction': out}
    except Exception as e: raise HTTPException(500, str(e))


print('✅ Tất cả endpoints OK!')

In [ ]:
# ══ CELL 6 · Start server + Ngrok ════════════════════════
import uvicorn, threading, time
from pyngrok import ngrok

NGROK_TOKEN = '3BM0sCxW0eDMEgAhshblG0Q6cP8_3rdRpdmFj8vkh8oojfZvA'

try:
    ngrok.kill()
    time.sleep(1)
except Exception:
    pass

ngrok.set_auth_token(NGROK_TOKEN)

def run():
    uvicorn.Server(uvicorn.Config(app, host='0.0.0.0', port=8000, log_level='warning')).run()

threading.Thread(target=run, daemon=True).start()
time.sleep(3)

tunnel = ngrok.connect(8000, bind_tls=True)
url    = tunnel.public_url

print()
print('=' * 65)
print('🎉  MEDPILOT ALL-IN-ONE SẴN SÀNG!')
print('=' * 65)
print(f'\n🌐 Ngrok URL: {url}\n')
print('📋 .env:')
print(f'VLLM_API_URL={url}/v1/chat/completions')
print(f'WHISPER_API_URL={url}/v1/audio/transcriptions')
print(f'EXTRACTION_API_URL={url}/v1/extract')
print(f'\n📋 frontend/.env.local:')
print(f'VITE_WHISPER_URL={url}/v1/audio/transcriptions')
print('\n⚠️  Giữ cell này chạy!')